# Verification via Google GenAI



In [69]:
from pathlib import Path
import pandas as pd


FHRS_DATA_DATE = "2026-07-23"

RANKED_NAMES_PATH = (Path("../data/business/interim") / f"london_fhrs_ranked_names_{FHRS_DATA_DATE}.csv")

ranked_names = pd.read_csv(RANKED_NAMES_PATH)

print(f"Unique ranked names: {len(ranked_names)}")

RANKED_FHRS_PATH = (Path("../data/business/interim") / f"london_fhrs_ranked_establishments_{FHRS_DATA_DATE}.csv")

fhrs_ranked = pd.read_csv(RANKED_FHRS_PATH)


Unique ranked names: 65799


# Exploring the dataset to find where to best use AI

In [71]:
score_thresholds = [
    0.99,
    0.95,
    0.90,
    0.80,
    0.70,
    0.60,
    0.50,
    0.40,
    0.30,
    0.20,
    0.10,
    0.05,
    0.02
]

threshold_counts = []

for threshold in score_thresholds:
    threshold_counts.append({"BakeryScore >= ": threshold, 
    "Names": (ranked_names["BakeryScore"].ge(threshold).sum())})

pd.DataFrame(threshold_counts)

,BakeryScore >=,Names
0,0.99,57
1,0.95,262
2,0.90,432
3,0.80,786
4,0.70,1151
5,0.60,1498
6,0.50,1847
7,0.40,2200
8,0.30,2607
9,0.20,3222


In [ ]:
score_bands = [(0.90, 1.01, "0.90-1.00"),
               (0.80, 0.90, "0.80-0.90"),
               (0.70, 0.80, "0.70-0.80"),
               (0.60, 0.70, "0.60-0.70"),
               (0.50, 0.60, "0.50-0.60"),
               (0.40, 0.50, "0.40-0.50"),
               (0.30, 0.40, "0.30-0.40"),
               (0.20, 0.30, "0.20-0.30"),
               (0.10, 0.20, "0.10-0.20"),
               (0.05, 0.10, "0.05-0.10")]

band_samples = []

for lower, upper, label in score_bands:
    band = ranked_names[ranked_names["BakeryScore"].between(lower, upper, inclusive="left")]

    sample = band.sample(n=min(5, len(band)), random_state=42).copy()
    sample["ScoreBand"] = label

    band_samples.append(sample)

score_band_sample = pd.concat(band_samples, ignore_index=True)

score_band_sample[["ScoreBand", "BakeryRank", "DisplayName", "BakeryScore", "StoreCount"]
                  ].sort_values(["ScoreBand", "BakeryScore"], ascending=[False, False])

,ScoreBand,BakeryRank,DisplayName,BakeryScore,StoreCount
3,0.90-1.00,31,MI CAKES LTD,0.995135,1
1,0.90-1.00,76,Dunns Bakery,0.987204,2
2,0.90-1.00,181,Wild Rise Bakery,0.969444,1
4,0.90-1.00,393,Bagel n Cakes,0.910481,1
0,0.90-1.00,425,Mario Ghost Bakery LTD,0.902067,1
6,0.80-0.90,475,Cakes By Sevi,0.888943,1
9,0.80-0.90,489,Pandan Bakery,0.884711,1
8,0.80-0.90,614,Rose Cake Creations,0.847638,1
5,0.80-0.90,653,"SUBA, SUBA Bakery",0.837160,1
7,0.80-0.90,719,Tee Cakes,0.819515,1


# Setting up AI API request

In [ ]:
api_key = getpass("Gemini API key: ")

client = genai.Client(api_key=api_key)

In [ ]:
# Choosen due ambigious name
TEST_RANK = 1474

test_business = (fhrs_ranked[fhrs_ranked["BakeryRank"] == TEST_RANK].iloc[0])

test_business[
    ["BakeryRank",
    "BusinessName",
    "BusinessType",
    "PostCode",
    "LocalAuthorityName",
    "BakeryScore",
    "StoreCount"]
]

BakeryRank                               1474
BusinessName                         No56 Ltd
BusinessType          Restaurant/Cafe/Canteen
PostCode                              SW4 7BG
LocalAuthorityName                    Lambeth
BakeryScore                          0.607385
StoreCount                                  1
Name: 1738, dtype: object

In [ ]:
prompt_1 = f"""
I'm checking whether this business should count as a bakery in a dataset of
London food establishments.

The business is:

Name: {test_business["BusinessName"]}
FHRS business type: {test_business["BusinessType"]}
Postcode: {test_business["PostCode"]}
Local authority: {test_business["LocalAuthorityName"]}

Please search the web and try to identify this specific business before making
a decision.

I want to count businesses whose main activity involves baking or specialist
retail of bread, pastries, cakes or similar baked goods. This includes
bakeries, patisseries and specialist cake businesses.

I do not want to count ordinary cafes, restaurants, supermarkets, convenience
stores or general food businesses just because they happen to sell baked goods.

If you cannot confidently identify the correct business or there is not enough
reliable evidence, say UNCLEAR rather than guessing.

Please make sure the evidence refers to this business and location where
possible, rather than another business with a similar name.

Give me:

VERDICT: BAKERY, NOT_BAKERY or UNCLEAR
REASON: a short explanation of what the business appears to do and why
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_1,
    config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
)

print(response.text)

VERDICT: NOT_BAKERY

REASON: The business, No56 Ltd at 56 Clapham Park Road, London SW4 7BG, is identified as a "Restaurant/Cafe/Canteen" by Food Hygiene Ratings. Further evidence indicates that it operates as a "No56 Wine and Tapas Bar," focusing on serving "delicious small plates, fine wines, and a warm, inviting atmosphere". Its offerings include "Mediterranean Tapas dishes and a full range of wines". This primary activity aligns with that of a restaurant or cafe, rather than a business whose main activity involves baking or specialist retail of bread, pastries, cakes, or similar baked goods.


In [ ]:
grounding = (response.candidates[0].grounding_metadata)

print("Searches:")
if grounding.web_search_queries:
    for query in grounding.web_search_queries:
        print("-", query)

print("\nSources:")
if grounding.grounding_chunks:
    for chunk in grounding.grounding_chunks:

        if chunk.web:
            print(f"- {chunk.web.title}\n"
                  f"  {chunk.web.uri}")

Searches:
- No56 Ltd SW4 7BG Lambeth business type
- No56 Ltd Clapham Common business
- What kind of business is No56 Ltd Lambeth?
- No56 Ltd restaurant cafe canteen SW4 7BG

Sources:
- food.gov.uk
  https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGUfo60PrGwd34lhlVL_ljPDrxLWeNPKqC29H_a8mCpLGPK2pon0cFnaGpJ1RNhhDD9WUrmdea-bG3RLrjaoC8P6cPU4T1dxLWnjf2P43ovdJvkl_4TQJtOtoc6DOx2hMLfJyxX196B8iqs2qR5lg1h
- food.gov.uk
  https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH4H5zkCy6AYuuq-7LNHSMynF5NnwgTeu81ZgnoNYAn3myjEzNAims1g1NarNwvFNNTnJFSn3E2oucVMn96o89ozQjkdwA-GCuaphMSpcY9fA97hnbJJFTQrV4q06mLkQquYaGWDrrYxOVE5JsY59ZsoCMK5sQZturBk6m8NWmC
- welcometofife.com
  https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHfG2VA7TrYGLyVH-VqCkPdtypNPvk3ADLGoX6ttlhO1JXleSr02FW-cibMrk0YeccvEkh0f-mLBVd_Yp2llN890H_txXXzXVlm0uxNMaFrDaeF9CJ_dEBdFUDVIK6UZaF32YhYz1bI0289HBVTvyiLkxsIbzhs9gc=
- crowdfunder.co.uk
  https://vertexaisearch.cloud.google.com/groundin

In [ ]:
test_businesses_v2 = (
    fhrs_ranked[fhrs_ranked["BakeryRank"].between(2000, 3000)
    ].drop_duplicates(subset="BusinessNameClean")
    .sample(n=10, random_state=2026)
    .sort_values("BakeryRank")
    .reset_index(drop=True))

test_businesses_v2[
    ["BakeryRank",
    "BusinessName",
    "BusinessType",
    "PostCode",
    "LocalAuthorityName",
    "BakeryScore"]
]

,BakeryRank,BusinessName,BusinessType,PostCode,LocalAuthorityName,BakeryScore
0,2151,Sarah's Cookies,Other catering premises,W3 0,Ealing,0.412723
1,2250,Sanam Sweet Treats,Retailers - other,E10,Waltham Forest,0.391217
2,2310,Sweetcations,Other catering premises,E9 5,Hackney,0.374707
3,2420,Greggs Ltd,Takeaway/sandwich shop,WC1V 6DR,Camden,0.349183
4,2437,Justcupcakes.Uk,Other catering premises,UB8,Hillingdon,0.345158
5,2498,MD Energy Limited,Retailers - other,N20 0LH,Barnet,0.326474
6,2578,Kokoro Bakes,Retailers - other,DA8,Bexley,0.306620
7,2655,Snacks & Sweets,Takeaway/sandwich shop,DA1 4JJ,Bexley,0.290176
8,2858,Nourishing Oven,Other catering premises,SE2,Greenwich,0.252117
9,2894,The quirky oven,Other catering premises,SE15,Southwark,0.246804


In [ ]:
business_text = ""

for i, row in test_businesses_v2.iterrows():
    business_text += f"""
    
BUSINESS {i + 1}
Name: {row["BusinessName"]}
FHRS type: {row["BusinessType"]}
Postcode: {row["PostCode"]}
Local authority: {row["LocalAuthorityName"]}

"""


prompt_2 = f"""
I'm checking whether each of the following London businesses should count as
a bakery in my dissertation dataset.

Please investigate EACH business separately using Google Search. Make sure
you identify the correct business and location rather than relying only on
the business name.

I count a business as BAKERY when baking or specialist retail of bread,
pastries, cakes or similar baked goods is a main part of what the business
does. This includes bakeries, patisseries and specialist cake businesses.

Use NOT_BAKERY when the business is mainly something else, such as a
restaurant, ordinary cafe, supermarket, convenience store, hotel or general
food retailer, even if it also sells baked goods.

Use UNCLEAR when you cannot confidently identify the correct business,
the evidence is weak or different sources disagree.

Do not skip any businesses and do not infer the answer from the business
name alone.

{business_text}

Return one result for EACH business in this exact format:

BUSINESS 1
VERDICT: BAKERY / NOT_BAKERY / UNCLEAR
REASON: short evidence-based explanation

BUSINESS 2
VERDICT: BAKERY / NOT_BAKERY / UNCLEAR
REASON: short evidence-based explanation

Continue this format through BUSINESS 10.
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_2,
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]))

print(response.text)

Here's the classification for each business:

**BUSINESS 1**
VERDICT: BAKERY
REASON: Sarah's Cookies is identified as a wholesale bakery specializing in a wide range of pastries and cookies.

**BUSINESS 2**
VERDICT: BAKERY
REASON: Sanam Sweets & Restaurant specializes in traditional Pakistani and Indian sweets (Mithai) and celebration cakes, which constitutes a main part of its retail offering alongside restaurant services.

**BUSINESS 3**
VERDICT: UNCLEAR
REASON: Searches for "Sweetcations E9 5 Hackney" did not confidently identify the correct business or provide sufficient evidence of its primary activity.

**BUSINESS 4**
VERDICT: BAKERY
REASON: Greggs describes itself as the "UK's leading bakery food-on-the-go retailer" and offers a wide range of baked goods, including savouries and sweet treats, as a core part of its business.

**BUSINESS 5**
VERDICT: BAKERY
REASON: Justcupcakes.Uk is a specialist cake business that focuses entirely on the creation and retail of cupcakes.

**BUSINE

In [ ]:
grounding = (response.candidates[0].grounding_metadata)

print("Searches:")
if grounding.web_search_queries:
    for query in grounding.web_search_queries:
        print("-", query)

print("\nSources:")
if grounding.grounding_chunks:
    for chunk in grounding.grounding_chunks:

        if chunk.web:
            print(f"- {chunk.web.title}\n"
                  f"  {chunk.web.uri}")

SEARCH QUERIES:

- Sarah's Cookies W3 0 Ealing
- Sanam Sweet Treats E10 Waltham Forest
- Sweetcations E9 5 Hackney
- Greggs Ltd WC1V 6DR Camden
- Justcupcakes.Uk UB8 Hillingdon
- MD Energy Limited N20 0LH Barnet
- Kokoro Bakes DA8 Bexley
- Snacks & Sweets DA1 4JJ Bexley
- Nourishing Oven SE2 Greenwich
- The quirky oven SE15 Southwark

SOURCES:

sarahscookiespdx.com
https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH8kHG3DEXmtNXW-mzU4ALcbO_AqPQ860Rk3YubalQ3x1WWVBcrirkXhLT1raEZtZ_2DcxsVSIXimBl-3Wjl36wjMmbtu4Lhm0-0DbkUEnsVHFhJLbyJMJIlUSh

sanamsweets.com
https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGjTkhOKlEJeetidsqs36xcPVZzS_j366OAuURoUVJLfIzusXr7ENg-kGZ411AVk4a18vNqU-2TRP-E0hgX2ozutKrsVaBWtDX2Nz3v6sS90On-pIK5tJXSmpqO60P7nQ==

sanamsweets.com
https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEUsoh5oZkxL1y3_ACJrfLkIIAGeQXX2vr9jReZK9kkBVae0MnlRpB41ZL0u3FC95l_1bcHo3-z99S5ElF63JZcM5gfkTBWyfwjX7r9vJlXOjWfIB96NTvBg7UNkzcc5aoiuhQRfpZ

In [ ]:
test_businesses_v3 = (
    fhrs_ranked[fhrs_ranked["BakeryRank"].between(2000, 3000)
    ].drop_duplicates(subset="BusinessNameClean")
    .sample(n=20, random_state=2027)
    .sort_values("BakeryRank")
    .reset_index(drop=True)
)

In [ ]:
business_text = ""

for i, row in test_businesses_v3.iterrows():
    business_text += f"""
    
BUSINESS {i + 1}
Name: {row["BusinessName"]}
FHRS type: {row["BusinessType"]}
Postcode: {row["PostCode"]}
Local authority: {row["LocalAuthorityName"]}

"""


prompt_3 = f"""
I'm checking whether each of the following London businesses should count as
a bakery in my dissertation dataset.

Please investigate EACH business separately using Google Search. Make sure
you identify the correct business and location rather than relying only on
the business name.

I count a business as BAKERY when baking or specialist retail of bread,
pastries, cakes or similar baked goods is a main part of what the business
does. This includes bakeries, patisseries and specialist cake businesses.

Use NOT_BAKERY when the business is mainly something else, such as a
restaurant, ordinary cafe, supermarket, convenience store, hotel or general
food retailer, even if it also sells baked goods.

Use UNCLEAR when you cannot confidently identify the correct business,
the evidence is weak or different sources disagree.

Do not skip any businesses. Investigate them independently and do not infer
one business from another.

{business_text}

Return one result for every business in order using:

BUSINESS [number]
VERDICT: BAKERY / NOT_BAKERY / UNCLEAR
REASON: short evidence-based explanation
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_3,
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]))

print(response.text)

BUSINESS 1
VERDICT: BAKERY
REASON: Happi Cheesecakes specializes in cheesecakes, indicating that baking is a main part of the business, even though it is currently undergoing a rebrand.BUSINESS 2
VERDICT: BAKERY
REASON: Mookies London Ltd primarily sells a variety of cookies, which are baked goods, and their menu clearly indicates a specialization in these items.BUSINESS 3
VERDICT: NOT_BAKERY
REASON: Veganlicious Sweets primarily supplies a wide range of gelatine-free and gluten-free confectionery (sweets/candies) like fizzy tongues and cola bottles, not baked goods such as bread, pastries, or cakes.BUSINESS 4
VERDICT: BAKERY
REASON: Although registered as a mobile caterer, "Portuguese treats limited" strongly suggests a specialization in traditional Portuguese baked goods such as pastéis de nata, cookies, and sweet bread, which are core bakery products.BUSINESS 5
VERDICT: NOT_BAKERY
REASON: My Personalised Sweets focuses on selling and personalizing confectionery items like flying sau

In [ ]:
grounding = (response.candidates[0].grounding_metadata)

print("Searches:")
if grounding.web_search_queries:
    for query in grounding.web_search_queries:
        print("-", query)

print("\nSources:")
if grounding.grounding_chunks:
    for chunk in grounding.grounding_chunks:

        if chunk.web:
            print(f"- {chunk.web.title}\n"
                  f"  {chunk.web.uri}")

SEARCH QUERIES:

- Happi Cheesecakes CR2 Croydon
- Happi Cheesecakes website
- Happi Cheesecakes menu
- Mookies London Ltd E1 6RU Tower Hamlets
- Mookies London Ltd products
- Mookies London Ltd business description
- Veganlicious Sweets UB6 Ealing
- Veganlicious Sweets products
- Veganlicious Sweets menu
- Portuguese treats limited HA2 Harrow
- Portuguese treats limited products
- Portuguese treats limited menu
- My Personalised Sweets SE12 Lewisham
- My Personalised Sweets products
- My Personalised Sweets menu
- Chouxru London ltd IG3 Redbridge
- Chouxru London ltd products
- Chouxru London ltd menu
- Chouxru London ltd desserts
- Robin Hood Cakes DA6 Bexley
- Robin Hood Cakes products
- Robin Hood Cakes menu
- Robin Hood Cakes reviews
- Mr & Mrs Sweet Bae SW16 Croydon
- Mr & Mrs Sweet Bae products
- Mr & Mrs Sweet Bae menu
- 100's Candy BR1 Bromley
- 100's Candy products
- 100's Candy menu
- 100's Candy reviews
- GAIL's Bakery Crystal Palace SE19 1RX Croydon
- GAIL's Bakery menu Cr

In [ ]:
test_businesses_v4 = (
    fhrs_ranked[fhrs_ranked["BakeryRank"].between(2000, 3000)
    ].drop_duplicates(subset="BusinessNameClean")
    .sample(n=10, random_state=2027)
    .sort_values("BakeryRank")
    .reset_index(drop=True))

test_businesses_v4[
    ["BakeryRank",
    "BusinessName",
    "BusinessType",
    "PostCode",
    "LocalAuthorityName",
    "BakeryScore"]
]

,BakeryRank,BusinessName,BusinessType,PostCode,LocalAuthorityName,BakeryScore
0,2156,Veganlicious Sweets,Manufacturers/packers,UB6,Ealing,0.410494
1,2321,Portuguese treats limited,Mobile caterer,HA2,Harrow,0.372286
2,2330,Chouxru London ltd,Other catering premises,IG3,Redbridge,0.370790
3,2472,100's Candy,Manufacturers/packers,BR1,Bromley,0.332202
4,2554,Candy Floss Creperie,Restaurant/Cafe/Canteen,E4 8DD,Waltham Forest,0.313261
5,2592,7th Heaven Foods Limited,Distributors/Transporters,NaN,City of London Corporation,0.302428
6,2853,Chim's Delight Ltd,Takeaway/sandwich shop,CR7 8RY,Croydon,0.252487
7,2904,Sweetasticldn,Retailers - other,BR4,Bromley,0.244499
8,2912,Cookie,Takeaway/sandwich shop,N14 4UT,Enfield,0.242816
9,2974,Ice Cream Hse Europe Ltd/K Desserts,Restaurant/Cafe/Canteen,E5 9ND,Hackney,0.235074


In [ ]:
business_text = ""

for i, row in test_businesses_v4.iterrows():
    business_text += f"""
    
BUSINESS {i + 1}
Name: {row["BusinessName"]}
FHRS type: {row["BusinessType"]}
Postcode: {row["PostCode"]}
Local authority: {row["LocalAuthorityName"]}

"""


prompt_4 = f"""
I'm checking whether each of these London food establishments should count as
a bakery in my dissertation dataset.

Please investigate EACH business independently using Google Search.

The most important requirement is that any evidence you use must refer to the
specific establishment supplied. A business with the same or a similar name
in another location is not valid evidence.

Before classifying a business, check whether the search evidence can reasonably
be matched to the supplied business using details such as its postcode, local
authority or London location.

If you cannot confidently match the evidence to the supplied establishment,
return UNCLEAR. Do not classify it from the business name alone.

A business counts as BAKERY when baking or specialist retail of bread,
pastries, cakes or similar baked goods is a main part of its actual activity.
This includes bakeries, patisseries and specialist cake businesses.

Use NOT_BAKERY when the establishment is mainly another type of business,
such as a restaurant, ordinary cafe, supermarket, convenience store, hotel,
general food retailer or catering business where baked goods are incidental.

Important:
- Do not infer what a business does from its name.
- Do not use evidence from similarly named businesses.
- Do not use evidence from a business in another city or country.
- The supplied postcode and local authority should be used to identify the
  establishment, not as optional information.
- If the location cannot be verified, use UNCLEAR.
- If reliable sources disagree, use UNCLEAR.
- It is better to return UNCLEAR than to guess.
- Investigate every business separately.

{business_text}

Return one result for every business in this format:

BUSINESS [number]
LOCATION_MATCH: YES / UNCLEAR
VERDICT: BAKERY / NOT_BAKERY / UNCLEAR
REASON: short explanation based only on evidence for the matched establishment
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_4,
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]))

print(response.text)

BUSINESS 1
LOCATION_MATCH: YES
VERDICT: NOT_BAKERY
REASON: Veganlicious Sweets primarily supplies a wide range of vegan confectionery/sweets, not baked goods.

BUSINESS 2
LOCATION_MATCH: YES
VERDICT: UNCLEAR
REASON: While the location is matched, there is no specific evidence to confirm that "Portuguese treats limited" primarily sells baked goods; the name is too general, and the business type "Mobile caterer" does not specify baked goods.

BUSINESS 3
LOCATION_MATCH: YES
VERDICT: BAKERY
REASON: The company's nature of business includes the manufacture of bread, fresh pastry goods, and cakes, and it lists pastries and cakes as a primary activity.

BUSINESS 4
LOCATION_MATCH: UNCLEAR
VERDICT: UNCLEAR
REASON: No specific information was found for "100's Candy" at the provided postcode and local authority to confidently determine its primary business activity.

BUSINESS 5
LOCATION_MATCH: YES
VERDICT: BAKERY
REASON: Candy Floss Creperie primarily focuses on the retail of crepes, waffles, cak

In [ ]:
grounding = (response.candidates[0].grounding_metadata)

print("Searches:")
if grounding.web_search_queries:
    for query in grounding.web_search_queries:
        print("-", query)

print("\nSources:")
if grounding.grounding_chunks:
    for chunk in grounding.grounding_chunks:

        if chunk.web:
            print(f"- {chunk.web.title}\n"
                  f"  {chunk.web.uri}")

SEARCH QUERIES:

- Veganlicious Sweets UB6 Ealing
- Portuguese treats limited HA2 Harrow
- Chouxru London ltd IG3 Redbridge
- 100's Candy BR1 Bromley
- Candy Floss Creperie E4 8DD Waltham Forest
- 7th Heaven Foods Limited City of London Corporation
- Chim's Delight Ltd CR7 8RY Croydon
- Sweetasticldn BR4 Bromley
- Cookie N14 4UT Enfield
- Ice Cream Hse Europe Ltd/K Desserts E5 9ND Hackney
- K Desserts E5 9ND Hackney

SOURCES:

veganlicioussweets.co.uk
https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFei2dTzZARcgjXrmztxJflEQRnJV6KnQTRmTjasOtT2DVmLykrI37JpqXbfoMDTO4tZIp1JrEseY2qiEce3hDpKViK3cgbghqgUbZdV-FzzdBmut9s9eZgZXJrzpVQWYQ=

veganlicioussweets.co.uk
https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHaTa7061LeDGGna8fGpnRivuqQ9AFWKPfkhqQa8B4yf8Uqskd0EKuAF1nj0QuRhpHXv-Aid3aRrNzla9R-nPY8mxX5WC7kBA0LN-uKpJFtA8BDt7uXUM0lJWTdH2OrON3Hz5xA

food.gov.uk
https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEZzHrWzHg-HhZdoZHcnz4Lw3HuHjprM0